# Ordered Logistic Regression Results for Adoption Predictors Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll use the `dataset.record_sets` property to enumerate all record sets in the dataset and display their `@id` values, along with their contained fields.

In [ ]:
# List all available record sets and their fields
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    print(f"  Fields:")
    for field in rs.fields:
        print(f"    - Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
    print('')

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

We use the record set and field `@id`s from the overview above to extract all data. We'll load each record set into a pandas DataFrame using their `@id`.

In [ ]:
# Prepare a dictionary of DataFrames keyed by record set @id
dataframes = {}

# Collect all record set @id values
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    print(f"Loading records for RecordSet @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")
    print(df.head(), '\n')

# For demonstration, pick the first record set to work with (update this as needed)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    main_df = dataframes[main_record_set_id]
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

The following example shows how to filter, normalize, and group numeric and categorical fields.

In [ ]:
# Example: Select a numeric field for processing (update as needed)
# Find available numeric fields
if record_set_ids:
    rs_obj = [rs for rs in dataset.record_sets if rs.id == main_record_set_id][0]
    numeric_field_candidates = [field for field in rs_obj.fields if field.data_type in ['schema:Float', 'schema:Number', 'schema:Integer']]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0].id
        numeric_field_name = numeric_field_candidates[0].name
        print(f"Using numeric field @id: {numeric_field_id}, name: {numeric_field_name}")

        # Filtering: arbitrary threshold
        threshold = main_df[numeric_field_name].mean() if main_df[numeric_field_name].dtype != object else 10
        filtered_df = main_df[main_df[numeric_field_name] > threshold]
        print(f"Filtered records where {numeric_field_name} > {threshold}")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_name}_normalized"] = (filtered_df[numeric_field_name] - filtered_df[numeric_field_name].mean()) / filtered_df[numeric_field_name].std()

        print(f"Normalized values for {numeric_field_name}:")
        print(filtered_df[[numeric_field_name, f"{numeric_field_name}_normalized"]].head())

        # Grouping by a categorical field (if present)
        group_field_candidates = [field for field in rs_obj.fields if field.data_type == 'schema:Text']
        if group_field_candidates:
            group_field_id = group_field_candidates[0].id
            group_field_name = group_field_candidates[0].name
            print(f"Grouping by categorical field @id: {group_field_id}, name: {group_field_name}")
            grouped_df = filtered_df.groupby(group_field_name)[numeric_field_name].mean().reset_index()
            print(grouped_df.head())
        else:
            print("No categorical text field found for grouping.")
    else:
        print("No numeric fields found in main record set.")
else:
    print("No record sets found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot a histogram of the numeric field used above, and a bar chart mean by categorical group if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for numeric field
if record_set_ids and numeric_field_candidates:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=main_df, x=numeric_field_name, kde=True)
    plt.title(f"Distribution of {numeric_field_name}")
    plt.xlabel(numeric_field_name)
    plt.ylabel('Frequency')
    plt.show()

# Bar chart by group (if grouping field found)
if 'grouped_df' in locals():
    plt.figure(figsize=(8, 4))
    sns.barplot(data=grouped_df, x=group_field_name, y=numeric_field_name)
    plt.title(f"Mean {numeric_field_name} by {group_field_name}")
    plt.xlabel(group_field_name)
    plt.ylabel(f"Mean {numeric_field_name}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR^2 dataset via its Croissant schema, explored available record sets and their fields using `mlcroissant`, extracted data, and performed simple analyses and visualizations using their `@id` for references. This process enables reproducibility and consistent referencing across the dataset. For further analysis, consider investigating relationships among other variables and deeper statistical explorations.